<a href="https://colab.research.google.com/drive/1R6fYYtk_kTUrFj_WWCskLl954BgZHkQP?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Hybrid (Ensamble) Search

Hybrid Search is a technique that combines multiple search methods to improve retrieval performance. Typically, it combines traditional keyword-based search (like BM25) with semantic search (using embeddings). This approach can provide better results than either method alone, especially for queries that have both keyword-specific and semantic aspects.

### Hybrid-search RAG Implementation:

1. **Hybrid Retrieval:** We use the EnsembleRetriever to get relevant documents using both keyword-based and semantic search.
2. **Response Generation:** Using the retrieved context, we generate a final response to the original query.
3. **Hybrid Search Explanation:** We generate an explanation of how the hybrid search process might have improved the retrieval of relevant information.

# Setup

1. **[LLM](https://groq.com/):** Groq's free Open source LLM endpoints([Groq API Key](https://console.groq.com/keys))
2. **[Vector Store](https://www.pinecone.io/learn/vector-database/):** [ChromaDB](https://www.trychroma.com/)
3. **[Embedding Model](https://qdrant.tech/articles/what-are-embeddings/):** [nomic-embed-text-v1.5](https://www.nomic.ai/blog/posts/nomic-embed-text-v1)
4. **[LLM Framework](https://python.langchain.com/v0.2/docs/introduction/):** LangChain
5. **[Huggingface API Key](https://huggingface.co/settings/tokens)**

# Install required libraries

In [11]:
"""
!pip install -q -U \
     Sentence-transformers==3.0.1 \
     langchain==0.3.19 \
     langchain-groq==0.2.4 \
     langchain-chroma==0.2.2 \
     langchain-community==0.3.18 \
     langchain-huggingface==0.1.2 \
     einops==0.8.1 \
     rank_bm25==0.2.2
"""     

'\n!pip install -q -U      Sentence-transformers==3.0.1      langchain==0.3.19      langchain-groq==0.2.4      langchain-chroma==0.2.2      langchain-community==0.3.18      langchain-huggingface==0.1.2      einops==0.8.1      rank_bm25==0.2.2\n'

### Import related libraries related to Langchain, HuggingfaceEmbedding

In [12]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma  import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

In [13]:
import getpass
import os

#### Provide a Groq API key. You can create one to access free open-source models at the following link.

[Groq API Creation Link](https://console.groq.com/keys)




In [14]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Provide Huggingface API Key. You can create Huggingface API key at following lin

[Higgingface API Creation Link](https://huggingface.co/settings/tokens)




In [15]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

### Step 1: Load and preprocess data code

In [16]:
def load_and_process_data(url):
    # Load data from web
    loader = WebBaseLoader(url)
    data = loader.load()

    # Split text into chunks (Experiment with Chunk Size and Chunk Overlap to get optimal chunking)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(data)

    return chunks

In [35]:
print(len(chunks))
chunks[201]

619


Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='With the increase of loneliness in the early 21st century, AI is sometimes identified as a potential source of relief to this problem. It would be possible, via human-like qualities built into AI products,[297] for individuals to assume that this need can be met by artificial means.[298][299] In some cases, people approach artificial intelligence for companionship when they believe that they would not find acceptance due to feeling outcast.[300] Examples of harm coming to humans from advanced')

### Step 2: Create vector store and BM25 retriever

In [17]:
def create_retrievers(chunks):
    embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})
    vectorstore = Chroma.from_documents(chunks, embeddings)
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 5  # Set number of documents to retrieve

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vectorstore.as_retriever(search_kwargs={"k": 5})],
        weights=[0.5, 0.5]
    )

    return ensemble_retriever

### Helper function for safe LLM calls

In [18]:
def safe_llm_call(prompt, **kwargs):
    try:
        response = llm.invoke(prompt.format(**kwargs))
        return response.content if response else "No response generated."
    except Exception as e:
        print(f"Error in LLM call: {e}")
        return "An error occurred while generating the response."

### Step 3: Hybrid-search RAG related code

1. **Hybrid Retrieval:** We use the EnsembleRetriever to get relevant documents using both keyword-based and semantic search.
2. **Response Generation:** Using the retrieved context, we generate a final response to the original query.
3. **Hybrid Search Explanation:** We generate an explanation of how the hybrid search process might have improved the retrieval of relevant information.

In [19]:
def hybrid_search_rag(query, ensemble_retriever, llm):
    # Hybrid retrieval
    retrieved_docs = ensemble_retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Generate response
    response_prompt = PromptTemplate.from_template(
        "You are an AI assistant tasked with answering questions based on the provided context. "
        "The context contains information retrieved using a hybrid search method combining keyword-based and semantic search. "
        "Please provide a comprehensive answer to the question, using the context when relevant "
        "and your general knowledge when necessary.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}\n"
        "Answer:"
    )
    final_answer = safe_llm_call(response_prompt, context=context, query=query)

    # Generate explanation of hybrid search process
    explanation_prompt = PromptTemplate.from_template(
        "Explain how the hybrid search process, combining keyword-based and semantic search, "
        "might have improved the retrieval of relevant information for answering the given query. "
        "Consider the potential benefits of this approach compared to using only one search method.\n\n"
        "Query: {query}\n"
        "Explanation:"
    )
    hybrid_search_explanation = safe_llm_call(explanation_prompt, query=query)

    return {
        "query": query,
        "final_answer": final_answer,
        "hybrid_search_explanation": hybrid_search_explanation,
        "retrieved_context": context
    }

### Step 4: Create chunk of web data to Chroma Vector Store

In [20]:

# Load and process data
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
chunks = load_and_process_data(url)

# Create ensemble retriever
ensemble_retriever = create_retrievers(chunks)

<All keys matched successfully>
[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


In [21]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5
)

response = llm.invoke([{"role": "user", "content": "Nation Animal of India"}])
response

AIMessage(content="The national animal of India is the **Bengal tiger (Panthera tigris tigris)**. It was declared the national animal in 1973, symbolizing strength, grace, and power, and it also highlights the country's commitment to wildlife conservation. The Bengal tiger is found primarily in the Indian subcontinent, especially in protected reserves such as Jim Corbett National Park, Ranthambore, and Sundarbans.", additional_kwargs={'reasoning_content': 'The user says "Nation Animal of India". Likely they want to know the national animal of India. Provide answer: Bengal tiger. Provide some info.'}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 75, 'total_tokens': 205, 'completion_time': 0.273026286, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.002750382, 'prompt_tokens_details': None, 'queue_time': 0.284172087, 'total_time': 0.275776668}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a42825ac', 'service_tie

### Step 5: Run Hybrid-Search RAG

This implementation shows the key parts of Hybrid-search RAG:

1. Combination of keyword-based (BM25) and semantic (vector) search for retrieval
2. Use of EnsembleRetriever to balance between different retrieval methods
3. Generation of a response using the hybrid-retrieved context
4. Explanation of the potential benefits of the hybrid search process

In [23]:
# Example queries
queries = [
        "What are the main applications of artificial intelligence in healthcare?",
        #"Explain the concept of machine learning and its relationship to AI.",
        #"Discuss the ethical implications of AI in decision-making processes."
    ]

# Run Hybrid-search RAG for each query
for query in queries:
  print(f"\nQuery: {query}")
  result = hybrid_search_rag(query, ensemble_retriever, llm)
  print("Final Answer:")
  print(result["final_answer"])
  print("\nHybrid Search Explanation:")
  print(result["hybrid_search_explanation"])
  print("\nRetrieved Context (first 300 characters):")
  print(result["retrieved_context"][:300] + "...")


Query: What are the main applications of artificial intelligence in healthcare?
Final Answer:
**Main applications of artificial intelligence (AI) in health‑care**

AI is now a core technology across many health‑care activities.  The literature and industry reports cited in the provided context (e.g., the *Artificial intelligence in healthcare* article and the Novartis blog on clinical‑data standards) highlight several broad categories in which AI is being deployed.  Below is a synthesis of those categories together with the most common concrete uses that are emerging today.

| Application area | Typical AI techniques | Representative uses in health‑care |
|------------------|----------------------|------------------------------------|
| **Clinical decision support & diagnosis** | Deep learning on imaging, natural‑language processing (NLP) of notes, probabilistic models | • Automated interpretation of radiology (X‑ray, CT, MRI) and pathology slides <br>• Early detection of diseases suc

In [25]:
result

{'query': 'What are the main applications of artificial intelligence in healthcare?',
 'final_answer': '**Main applications of artificial intelligence (AI) in health‑care**\n\nAI is now a core technology across many health‑care activities.  The literature and industry reports cited in the provided context (e.g., the *Artificial intelligence in healthcare* article and the Novartis blog on clinical‑data standards) highlight several broad categories in which AI is being deployed.  Below is a synthesis of those categories together with the most common concrete uses that are emerging today.\n\n| Application area | Typical AI techniques | Representative uses in health‑care |\n|------------------|----------------------|------------------------------------|\n| **Clinical decision support & diagnosis** | Deep learning on imaging, natural‑language processing (NLP) of notes, probabilistic models | • Automated interpretation of radiology (X‑ray, CT, MRI) and pathology slides <br>• Early detection 